In [1]:
import chromadb
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
client = chromadb.PersistentClient(path="../chroma_db")

print("Model ve veritabani hazir!")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model ve veritabani hazir!


In [2]:
collection = client.get_or_create_collection(name="knee_passages")

print("Koleksiyon hazir!")
print("Icindeki pasaj sayisi:", collection.count())

Koleksiyon hazir!
Icindeki pasaj sayisi: 0


In [3]:
passages = [
    "Most patients begin moving around with a walker the day after the operation.",
    "Physical therapy helps strengthen the muscles around your new knee.",
    "Knee replacement is a common procedure for severe arthritis.",
    "Most people stay in hospital for 1 to 3 days after knee replacement surgery.",
    "Blood clots are a possible risk after knee replacement surgery.",
    "The weather in London is rainy today.",
]

metadatas = [
    {"source": "NHS (ornek)", "topic": "recovery"},
    {"source": "MedlinePlus (ornek)", "topic": "rehabilitation"},
    {"source": "NIH NIAMS (ornek)", "topic": "general"},
    {"source": "NHS (ornek)", "topic": "hospital stay"},
    {"source": "MedlinePlus (ornek)", "topic": "risks"},
    {"source": "Test", "topic": "unrelated"},
]

ids = ["p1", "p2", "p3", "p4", "p5", "p6"]

embeddings = model.encode(passages).tolist()

collection.upsert(
    ids=ids,
    documents=passages,
    embeddings=embeddings,
    metadatas=metadatas,
)

print("Eklendi! Pasaj sayisi:", collection.count())




Eklendi! Pasaj sayisi: 6


In [4]:
question = "When can I walk after knee surgery?"

results = collection.query(
    query_embeddings=[model.encode(question).tolist()],
    n_results=3,
)

for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    score = 1 - dist / 2
    print(f"{score:.2f} | {meta['source']} | {doc}")

0.68 | NHS (ornek) | Most people stay in hospital for 1 to 3 days after knee replacement surgery.
0.59 | NHS (ornek) | Most patients begin moving around with a walker the day after the operation.
0.53 | MedlinePlus (ornek) | Physical therapy helps strengthen the muscles around your new knee.


In [5]:
question = "What is the capital of France?"

results = collection.query(
    query_embeddings=[model.encode(question).tolist()],
    n_results=3,
)

for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    score = 1 - dist / 2
    print(f"{score:.2f} | {meta['source']} | {doc}")

0.09 | Test | The weather in London is rainy today.
-0.06 | MedlinePlus (ornek) | Physical therapy helps strengthen the muscles around your new knee.
-0.07 | NHS (ornek) | Most patients begin moving around with a walker the day after the operation.


# Deneyden Çıkan Sonuçlar

## 1. Vektör veritabanı ne işe yarar?
Pasajlar bir kez vektöre çevrilip ChromaDB'de kalıcı olarak saklandı. Her pasajla birlikte **metni** ve **kaynağı** da saklandı. Soru geldiğinde ChromaDB en yakın pasajları kaynaklarıyla birlikte getirdi. Sistemin kaynak gösterebilmesi bu sayede mümkün.

## 2. ChromaDB sonuçları dünkü deneyle tutarlı
"Yürüteç" pasajı dün de bugün de **0.59** aldı. ChromaDB, elle yaptığımız hesabın aynısını yapıyor.

## 3. En yüksek skor her zaman doğru cevap değil
"Ne zaman yürüyebilirim?" sorusunda birinci sırada **0.68** ile "hastanede 1-3 gün kalınır" pasajı çıktı. Bu pasaj soruyu cevaplamıyor ama soruyla çok ortak ifade paylaşıyor. Doğru cevap ikinci sırada kaldı (**0.59**).

Bundan çıkan dersler:
- Sadece en yüksek skorlu pasaja güvenilemez, birkaç pasaj getirilmeli (**Top-K**, K > 1).
- Hangi pasajın soruyu gerçekten cevapladığına LLM karar vermeli.
- Bu durum, LLM'in yanlış pasajı kaynak olarak göstermesi riskini (**Citation Correctness**) somut olarak gösteriyor. Değerlendirmede sadece cevap değil, gösterilen kaynak da kontrol edilmeli.

## 4. Alakasız sorularda abstain kolay
"Fransa'nın başkenti neresidir?" sorusunda en yüksek skor sadece **0.09** oldu. Diz sorusunda ise **0.68** idi. Aradaki büyük fark sayesinde sistem bu tür sorularda kolayca cevap vermekten kaçınabilir.

## 5. "Cevap yok" kararını veritabanı vermez
ChromaDB, soru ne kadar alakasız olursa olsun her zaman en yakın K pasajı getirir. Cevap verip vermeme kararını bizim yazacağımız **eşik kontrolü** verecek.

## 6. ChromaDB kullanımı
Kurulum ve kullanım çok kolaydı: ayrı bir sunucu gerekmedi, birkaç satır kodla veri eklendi ve arama yapıldı. Bu deneyim, ChromaDB'yi geçici tercih olarak seçme kararını destekliyor.
